# 01 下载 Qwen 权重（本地 Mac）

**已核实的 ModelScope 仓库名**（2026-10-16 实测）：

| 用途 | 仓库 ID | 大小 |
|---|---|---|
| SFT 基座 | `Qwen/Qwen3-4B-Instruct-2507` | ~8.5GB |
| 打标/兜底 | `Qwen/Qwen3-32B-AWQ` | ~19GB |
| 备选基座 | `Qwen/Qwen3-8B`（base） | ~16GB |
| 端侧 GGUF | `Qwen/Qwen3-1.7B-GGUF` / `Qwen/Qwen3-0.6B-GGUF` | ~1.2/0.5GB |

注意：`Qwen/Qwen3-4B-Instruct`（无 2507）在 ModelScope 上**不存在**，404 就是这个原因；8B 的 Instruct 版 ModelScope 也没有，用 base 或走 hf-mirror。

In [1]:
import os
LOCAL = os.path.expanduser("~/models/parallax")
os.makedirs(LOCAL, exist_ok=True)
print("下载到:", LOCAL)

下载到: /Users/chris/models/parallax


In [2]:
# 通用下载函数：ModelScope 优先，失败自动切 hf-mirror，断点续传
import time

def download_model(repo_id, dst):
    """ModelScope 下载，3 次重试；仓库不存在则提示改走 hf-mirror。"""
    from modelscope.hub.snapshot_download import snapshot_download
    last = None
    for i in range(3):
        try:
            return snapshot_download(repo_id, local_dir=dst)
        except Exception as e:
            last = e; print(f"[{i+1}/3] 重试: {repo_id} -> {type(e).__name__}")
            time.sleep(5)
    raise RuntimeError(f"ModelScope 下载失败 {repo_id}: {last}")

def download_hf(repo_id, dst, filename=None):
    """hf-mirror 备用源（需先执行过 00 notebook 的镜像配置）。"""
    os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
    from huggingface_hub import snapshot_download, hf_hub_download
    if filename:
        return hf_hub_download(repo_id, filename, local_dir=dst)
    return snapshot_download(repo_id, local_dir=dst)

In [ ]:
# P0：SFT 基座 Qwen3-4B-Instruct-2507（~8.5GB）
dst = os.path.join(LOCAL, "Qwen3-4B-Instruct-2507")
try:
    download_model("Qwen/Qwen3-4B-Instruct-2507", dst)
except RuntimeError as e:
    print(e, "\n改用 hf-mirror ...")
    download_hf("Qwen/Qwen3-4B-Instruct-2507", dst)

/Users/chris/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/chris/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


2026-10-16 22:24:58,501 - modelscope - INFO - Got 13 files, start to download ...
Processing 13 items:   0%|          | 0.00/13.0 [00:00<?, ?it/s]


















































Processing 13 items:   8%|▊         | 1.00/13.0 [00:01<00:13, 1.15s/it]
































































Processing 13 items:  15%|█▌        | 2.00/13.0 [00:01<00:09, 1.14it/s]

























































Processing 13 items:  31%|███       | 4.00/13.0 [00:02<00:04, 2.08it/s]
















































Processing 13 items:  46%|████▌     | 6.00/13.0 [00:02<00:02, 2.90it/s]












Processing 13 items:  54%|█████▍    | 7.00/13.0 [00:02<00:01, 3.45it/s]
















































































































Processing 13 items:  62%|██████▏   | 8.00/13.0 [00:03<00:01, 2.55it/s]




























Processing 13 items:  69%|██████▉   | 9.00/13.0 [0

In [ ]:
# P0：自托管打标 Qwen3-32B-AWQ（~19GB，最大，建议最后跑）
dst = os.path.join(LOCAL, "Qwen3-32B-AWQ")
download_model("Qwen/Qwen3-32B-AWQ", dst)

In [ ]:
# P1：备选基座 Qwen3-8B（base，~16GB；ModelScope 无 8B Instruct）
dst = os.path.join(LOCAL, "Qwen3-8B")
download_model("Qwen/Qwen3-8B", dst)

In [ ]:
# P2：端侧 GGUF（ModelScope 官方 GGUF 仓库）
for repo in ["Qwen/Qwen3-1.7B-GGUF", "Qwen/Qwen3-0.6B-GGUF"]:
    dst = os.path.join(LOCAL, repo.split("/")[-1])
    download_model(repo, dst)
    for f in sorted(os.listdir(dst)):
        if f.endswith(".gguf"):
            print(repo.split("/")[-1], f, round(os.path.getsize(os.path.join(dst,f))/1e9,2), "GB")

In [ ]:
# 下载完成后的大小清单
import pathlib
for p in sorted(pathlib.Path(LOCAL).iterdir()):
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
    print(f"{p.name:24s} {size:7.2f} GB")